# Tool catalog: file, playback, and clipboard

Run these cells in order in a disposable JupyterLab project with its live kernel. The blue PNG and quiet 60-second WAV are generated inputs. The longer clip gives you time to try the playback controls. Click the file picker, Play, Copy, or paste control when it appears; no browser can perform those user actions silently. Each call returns an immediate receipt, so inspect it on a later turn after the media row completes. The AI questions ask the selected ChatGPT connection to make real, scoped tool calls. Do not run all cells at once.


## Disposable exact media references

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib, math, struct, wave
from PIL import Image
from nbinlineai.tools import (choose_file, open_media, play_media, pause_media, seek_media, set_media_volume, close_media, copy_text, paste_content, operation_status, release_media)
from IPython.display import display, Audio

server_root = Path.cwd().resolve()
nonce = uuid4().hex
image_path = server_root / f'playback-{nonce}.png'
tone_path = server_root / f'playback-{nonce}.wav'
Image.new('RGB', (2, 2), (36, 112, 195)).save(image_path)
with wave.open(str(tone_path), 'wb') as output:
    output.setnchannels(1); output.setsampwidth(2); output.setframerate(16000)
    output.writeframes(b''.join(struct.pack('<h', int(1200 * math.sin(2 * math.pi * 440 * i / 16000))) for i in range(60 * 16000)))
image_ref = {'path': image_path.name, 'sha256': hashlib.sha256(image_path.read_bytes()).hexdigest()}
tone_ref = {'path': tone_path.name, 'sha256': hashlib.sha256(tone_path.read_bytes()).hexdigest()}
print('Generated quiet 60-second 440 Hz WAV:', tone_path.name)
print('Generated blue PNG:', image_path.name)
display(Image.open(image_path).resize((80, 80)))
display(Audio(filename=str(tone_path), normalize=False))


## choose_file

In [ ]:
from nbinlineai.tools import choose_file
selected = choose_file(accept='image/png')
selected  # In the visible chooser select only the generated playback-*.png sample.


In [ ]:
print('Chosen file:', selected.status, selected.error)
if selected.status == 'completed':
    print('Imported image:', {key: selected.media.get(key) for key in ('media_id', 'mime_type', 'sha256')})
    display(selected.result.resize((80, 80)))


Use &`choose_file` with `accept='image/png'` so I can select this notebook's generated blue PNG in the visible picker. Report the immediate operation ID/state only; do not choose a personal file or claim a hash before selection completes.


After I select the generated PNG, use &`operation_status` on the `choose_file` operation ID. Report actual MIME type, SHA-256, and managed media ID only if completed; otherwise retry later.


Use &`release_media` only on the managed media ID from the completed AI file-choice status. Report the immediate release operation ID/state; the displayed Python image remains available.


Use &`operation_status` on the AI `release_media` operation ID after it finishes. Report its actual terminal state; the generated source PNG still exists.


## open_media

In [ ]:
from nbinlineai.tools import open_media
opened = open_media(tone_ref)
opened  # The owned preview uses the exact WAV bytes and hash.


In [ ]:
print('Open preview:', opened.status, opened.error)
if opened.status == 'completed':
    preview_id = opened.result['preview_id']
    print('Preview ID:', preview_id)


Use &`open_media` with the exact quiet 60-second WAV reference $`tone_ref`. Report its immediate operation ID/state; do not claim a preview ID until the later status question confirms it.


Use &`operation_status` on the AI `open_media` operation ID after completion. Report its actual preview ID and media type; if still running, stop and retry later.


Use &`close_media` only on the new preview ID from the preceding completed AI open status. Report the close operation ID/state; leave the direct Python preview open for later controls.


Use &`operation_status` on that AI `close_media` operation ID after completion. Confirm only the extra AI preview closed.


## play_media

In [ ]:
from nbinlineai.tools import play_media
assert opened.status == 'completed', 'Wait for the preview, then rerun this cell.'
playing = play_media(opened.result['preview_id'])
playing  # Click Play in the visible panel if the browser requires activation.


In [ ]:
print('Play:', playing.status, playing.result, playing.error)


Use &`play_media` on the direct preview ID $`preview_id`. Report its immediate operation ID/state. I will click the visible Play control if the browser waits for activation.


After any visible Play click, use &`operation_status` on that `play_media` operation ID. Report actual playback state only if completed; otherwise retry later.


## pause_media

In [ ]:
from nbinlineai.tools import pause_media
paused = pause_media(opened.result['preview_id'])
paused


In [ ]:
print('Pause:', paused.status, paused.result, paused.error)


Use &`pause_media` on preview ID $`preview_id`. Report its immediate operation ID/state, not a final paused state.


Use &`operation_status` on that `pause_media` operation ID after completion. Report the actual paused state.


## seek_media

In [ ]:
from nbinlineai.tools import seek_media
seeked = seek_media(opened.result['preview_id'], 2.5)
seeked


In [ ]:
print('Seek:', seeked.status, seeked.result, seeked.error)


Use &`seek_media` on preview ID $`preview_id` at 2.5 seconds of the generated 60-second tone. Report its immediate operation ID/state, not the final actual position.


Use &`operation_status` on that `seek_media` operation ID after completion. Report actual position, which can differ slightly from the requested 2.5 seconds.


## set_media_volume

In [ ]:
from nbinlineai.tools import set_media_volume
volume = set_media_volume(opened.result['preview_id'], 0.25)
volume


In [ ]:
print('Volume:', volume.status, volume.result, volume.error)


Use &`set_media_volume` on preview ID $`preview_id` with volume 0.25. Report its immediate operation ID/state; the browser may restrict volume changes.


Use &`operation_status` on that `set_media_volume` operation ID. Report the actual result or browser restriction without claiming the volume changed if it did not.


## close_media

In [ ]:
from nbinlineai.tools import close_media
closed = close_media(opened.result['preview_id'])
selected_release = release_media(selected.media['media_id']) if selected.status == 'completed' and selected.media.get('media_id') else None
closed


In [ ]:
print('Close preview:', closed.status, closed.result, closed.error)
print('Release selected PNG:', selected_release.status, selected_release.error) if selected_release else print('No managed PNG to release.')
print('Original 60-second WAV remains:', tone_path.exists())


Use &`open_media` on $`tone_ref` to create a fresh preview for the AI close example. Report its immediate operation ID/state; wait for its preview ID before closing it.


Use &`operation_status` on the fresh AI `open_media` operation ID. Report its preview ID only if completed; otherwise retry later.


Use &`close_media` on that fresh AI preview ID. Report the immediate close operation ID/state; the generated WAV file remains intact.


Use &`operation_status` on that final AI `close_media` operation ID after completion. Report the actual close state and whether the generated source WAV remains.


## copy_text

In [ ]:
from nbinlineai.tools import copy_text
copied = copy_text('nbinlineai disposable playback sample')
copied  # Click the visible Copy button or use the browser copy shortcut.


In [ ]:
print('Copy:', copied.status, copied.result, copied.error)


Use &`copy_text` with the literal `nbinlineai disposable playback sample`. Report its immediate operation ID/state. Ask me to click the visible Copy control if the browser waits.


After I click Copy, use &`operation_status` on that `copy_text` operation ID. Report the actual terminal state; do not claim success from an initial waiting snapshot.


## paste_content

In [ ]:
from nbinlineai.tools import paste_content
pasted = paste_content(accept='text')
pasted  # Click the visible paste box and paste the sample text.


In [ ]:
print('Paste:', pasted.status, pasted.error)
if pasted.status == 'completed':
    print('Pasted text:', pasted.result)


Use &`paste_content` with `accept='text'`. Report its immediate operation ID/state. Ask me to paste the disposable sample into the visible paste box; do not read the clipboard silently.


After I paste the disposable text, use &`operation_status` on that `paste_content` operation ID. Quote only its actual bounded returned text and truncation flag if completed; otherwise retry later.


In [ ]:
for generated in (tone_path, image_path):
    generated.unlink(missing_ok=True)
print('Disposable playback samples removed:', not tone_path.exists() and not image_path.exists())
if globals().get('selected_release'):
    print('Imported image release:', selected_release.status, selected_release.error)
